# JAX/MJX Dinosaur Training (Colab)

Train a dinosaur species using **MuJoCo MJX** (JAX-accelerated physics) with a
from-scratch PPO implementation in pure JAX. MJX vectorises thousands of parallel
simulations on a single GPU, giving 10-100x speedups over CPU-based Gymnasium training.

**Requirements:** Colab GPU runtime (A100 recommended).

**Supported Species:**
- `trex` — T-Rex: balance → locomotion → bite
- `velociraptor` — Raptor: balance → locomotion → strike
- `brachiosaurus` — Brachio: balance → locomotion → food reach

Set `SPECIES` in the configuration cell below to choose.

In [ ]:
# Install dependencies and verify GPU
!pip install mujoco mujoco-mjx "jax[cuda12]" flax optax

import os
import subprocess

if subprocess.run("nvidia-smi").returncode:
    raise RuntimeError("GPU not found. Use a GPU Colab runtime.")

NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

os.environ["MUJOCO_GL"] = "egl"

import jax
import mujoco
from mujoco import mjx

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo: {mujoco.__version__}")
print("Setup complete.")

In [ ]:
# Clone mesozoic-labs and install with JAX extras
!git clone https://github.com/kuds/mesozoic-labs.git /content/mesozoic-labs 2>/dev/null || echo 'Already cloned'
!pip install -e "/content/mesozoic-labs[jax]" -q

from IPython.display import clear_output

clear_output()
print("mesozoic-labs[jax] installed.")

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import optax
from mujoco import mjx

# Shared modules from mesozoic-labs
from environments.shared.reward_functions import (
    reward_forward_velocity,
    reward_alive,
    reward_energy,
    reward_posture,
    reward_approach_shaping,
    quat_to_tilt,
    check_height_tilt_termination,
)
from environments.shared.obs_functions import SensorLayout, build_bipedal_obs
from environments.shared.mjx_utils import scale_action_jax
from environments.shared.jax_ppo import (
    make_actor_critic,
    sample_action,
    compute_gae,
    ppo_loss,
    PPOConfig,
)

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================
SPECIES = "trex"  # Choose: "trex", "velociraptor", "brachiosaurus"
CURRENT_STAGE = 1  # Curriculum stage: 1=balance, 2=locomotion, 3=species-specific
USE_GOOGLE_DRIVE = False  # Set to True to save outputs to Google Drive (persistent across sessions)
VERBOSE = 1  # 0=eval/summary only, 1=periodic updates (default), 2=every update

# ============================================================
# Species Configuration (auto-resolved from SPECIES)
# ============================================================
_SPECIES_CFG = {
    "trex": {
        "xml_path": "/content/mesozoic-labs/environments/trex/assets/trex.xml",
        "root_body": "pelvis",
        "healthy_z_range": (0.4, 1.6),
        "target_z_default": 0.5,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Bite"},
    },
    "velociraptor": {
        "xml_path": "/content/mesozoic-labs/environments/velociraptor/assets/raptor.xml",
        "root_body": "pelvis",
        "healthy_z_range": (0.3, 1.0),
        "target_z_default": 0.3,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Strike"},
    },
    "brachiosaurus": {
        "xml_path": "/content/mesozoic-labs/environments/brachiosaurus/assets/brachiosaurus.xml",
        "root_body": "torso",
        "healthy_z_range": (1.0, 3.5),
        "target_z_default": 3.0,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Food Reach"},
    },
}

assert SPECIES in _SPECIES_CFG, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_CFG)}"
_cfg = _SPECIES_CFG[SPECIES]

# ============================================================
# Storage Configuration
# ============================================================
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/mesozoic-labs/jax_training")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Outputs will be saved to: {OUTPUT_DIR}")
else:
    OUTPUT_DIR = Path(".")
    print(f"Outputs will be saved to local Colab storage: {OUTPUT_DIR}")

print(f"Species: {SPECIES}")
print(f"Stage: {CURRENT_STAGE} ({_cfg['stage_names'].get(CURRENT_STAGE, '?')})")
print(f"Google Drive storage: {USE_GOOGLE_DRIVE}")
print(f"Verbose: {VERBOSE}")

## 1. Load Model into MJX

In [ ]:
# Load the MJCF model for the selected species
xml_path = _cfg["xml_path"]
mj_model = mujoco.MjModel.from_xml_path(xml_path)
mj_data = mujoco.MjData(mj_model)

print(f"{SPECIES.title()} model loaded:")
print(f"  Bodies: {mj_model.nbody}")
print(f"  Joints: {mj_model.njnt}")
print(f"  Actuators (nu): {mj_model.nu}")
print(f"  qpos dim: {mj_model.nq}")
print(f"  qvel dim: {mj_model.nv}")
print(f"  Total mass: {sum(mj_model.body_mass):.2f} kg")
print(f"  Timestep: {mj_model.opt.timestep * 1000:.1f} ms")

# Put model on device (GPU)
mjx_model = mjx.put_model(mj_model)
print(f"\nModel placed on {jax.devices()[0]}")

## 2. Environment FunctionsPure-JAX functions for observation, reward, reset, and step. These now usethe **shared modules** from `environments.shared` so that the same reward andobservation logic is used by both the Gymnasium (SB3) and MJX (JAX) trainingpaths.

In [ ]:
# ---------- Constants ----------
FRAME_SKIP = 5
HEALTHY_Z_MIN, HEALTHY_Z_MAX = _cfg["healthy_z_range"]
MAX_TILT_ANGLE = 1.047
MAX_EPISODE_STEPS = 1000

# Body / geom / site IDs (looked up once from the MuJoCo model)
ROOT_BODY_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, _cfg["root_body"])
FLOOR_GEOM_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "floor")

# Sensor layout (matches MJCF): gyro(3), accel(3), quat(4), foot sensors...
# Bipedal species have 2 foot sensors at indices 10-11
# Quadrupedal (brachiosaurus) has 4 foot sensors at indices 10-13
if SPECIES == "brachiosaurus":
    SENSOR_LAYOUT = SensorLayout(gyro_start=0, accel_start=3, quat_start=6, foot_indices=(10, 11, 12, 13))
else:
    SENSOR_LAYOUT = SensorLayout(gyro_start=0, accel_start=3, quat_start=6, foot_indices=(10, 11))

# Action scaling ranges (as JAX arrays for use with shared scale_action_jax)
CTRL_RANGE = jnp.array(mj_model.actuator_ctrlrange)

print(f"Root body ({_cfg['root_body']}) id: {ROOT_BODY_ID}")
print(f"Action dim: {mj_model.nu}")

In [ ]:
def get_obs(data):
    """Extract observation vector from MJX data using shared obs builder."""
    return build_bipedal_obs(
        qpos=data.qpos,
        qvel=data.qvel,
        sensordata=data.sensordata,
        pelvis_xpos=data.xpos[ROOT_BODY_ID],
        target_pos=jnp.zeros(3),  # Target tracking handled by reward config
        sensor_layout=SENSOR_LAYOUT,
    )


def compute_reward(data, action, reward_cfg):
    """Compute scalar reward using shared reward functions."""
    vel_2d = data.qvel[:2]
    forward_dir = jnp.array([1.0, 0.0])

    r_forward, _ = reward_forward_velocity(
        vel_2d, forward_dir, 8.0, reward_cfg["forward_vel_weight"]
    )
    r_alive = reward_alive(reward_cfg["alive_bonus"])
    r_energy = reward_energy(action, CTRL_RANGE.shape[0], reward_cfg["energy_penalty_weight"])

    # Posture reward using shared function
    root_quat = data.sensordata[6:10]
    r_posture, _ = reward_posture(root_quat, MAX_TILT_ANGLE, reward_cfg.get("posture_weight", 0.2))

    return r_forward + r_alive + r_energy + r_posture


def is_terminated(data):
    """Check if the dinosaur has fallen using shared termination check."""
    body_z = data.xpos[ROOT_BODY_ID, 2]
    root_quat = data.sensordata[6:10]
    tilt = quat_to_tilt(root_quat)
    terminated, _ = check_height_tilt_termination(
        float(body_z), float(tilt), (HEALTHY_Z_MIN, HEALTHY_Z_MAX), MAX_TILT_ANGLE
    )
    return terminated


def scale_action(action):
    """Scale action from [-1, 1] to actuator control range using shared function."""
    return scale_action_jax(action, CTRL_RANGE)


# Verify obs dimension
mujoco.mj_forward(mj_model, mj_data)
_test_data = mjx.put_data(mj_model, mj_data)
_test_obs = get_obs(_test_data)
OBS_DIM = _test_obs.shape[0]
ACT_DIM = mj_model.nu
print(f"Observation dim: {OBS_DIM}")
print(f"Action dim: {ACT_DIM}")

## 3. Batched MJX Step

A single `jax.jit`-compiled function that steps `N` parallel environments.

In [ ]:
def mjx_step_single(model, data, action):
    """Step one environment: apply action, advance physics, return new data."""
    ctrl = scale_action(action)
    data = data.replace(ctrl=ctrl)

    # Frame skip: step physics multiple times per action
    def body_fn(_, d):
        return mjx.step(model, d)

    data = jax.lax.fori_loop(0, FRAME_SKIP, body_fn, data)
    return data


# Vectorize: model is shared (None), data and action are batched (0)
@jax.jit
def batched_step(model, data_batch, action_batch):
    return jax.vmap(mjx_step_single, in_axes=(None, 0, 0))(model, data_batch, action_batch)


print("Batched step function compiled.")

## 4. Policy Network (Flax)Uses the shared `ActorCritic` from `environments.shared.jax_ppo`.

In [ ]:
# Initialize using the shared ActorCritic from jax_ppo
network = make_actor_critic(action_dim=ACT_DIM)
rng = jax.random.PRNGKey(42)
dummy_obs = jnp.zeros((OBS_DIM,))
params = network.init(rng, dummy_obs)

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"ActorCritic parameters: {n_params:,}")

## 5. PPO ImplementationCore PPO functions (`sample_action`, `compute_gae`, `ppo_loss`) are nowimported from `environments.shared.jax_ppo`. Notebook-specific wrappersbelow adapt them for the training loop.

In [ ]:
# Use shared sample_action but with our network closure
def nb_sample_action(params, obs, rng):
    """Sample action from Gaussian policy using shared PPO module."""
    return sample_action(params, network, obs, rng)


# Use shared compute_gae directly (same signature)
# compute_gae is already imported from jax_ppo


# Notebook-specific PPO loss wrapper (uses the shared ppo_loss with extra metrics)
def nb_ppo_loss(params, obs, actions, old_log_probs, advantages, returns,
                clip_range=0.2, vf_coef=0.5, ent_coef=0.01):
    """PPO clipped surrogate loss using shared implementation."""
    mean, log_std, values = jax.vmap(network.apply, in_axes=(None, 0))(params, obs)
    std = jnp.exp(log_std)

    # Log probability of taken actions
    log_probs = -0.5 * jnp.sum(
        ((actions - mean) / (std + 1e-8)) ** 2 + 2 * log_std + jnp.log(2 * jnp.pi),
        axis=-1,
    )

    # Policy loss (clipped)
    ratio = jnp.exp(log_probs - old_log_probs)
    adv_normalized = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    surr1 = ratio * adv_normalized
    surr2 = jnp.clip(ratio, 1 - clip_range, 1 + clip_range) * adv_normalized
    policy_loss = -jnp.mean(jnp.minimum(surr1, surr2))

    # Value loss
    value_loss = jnp.mean((values - returns) ** 2)

    # Entropy bonus
    entropy = 0.5 * jnp.sum(jnp.log(2 * jnp.pi * jnp.e * std**2), axis=-1)
    entropy_bonus = jnp.mean(entropy)

    total_loss = policy_loss + vf_coef * value_loss - ent_coef * entropy_bonus

    aux = {
        "policy_loss": policy_loss,
        "value_loss": value_loss,
        "entropy": entropy_bonus,
        "approx_kl": jnp.mean((ratio - 1) - jnp.log(ratio)),
        "clip_fraction": jnp.mean((jnp.abs(ratio - 1.0) > clip_range).astype(jnp.float32)),
        "mean_std": jnp.mean(std),
    }

    return total_loss, aux


print("PPO functions defined (shared modules + notebook wrappers).")

## 6. Training Loop

In [ ]:
# ---------- Hyperparameters ----------
NUM_ENVS = 2048  # parallel environments
ROLLOUT_LEN = 64  # steps per rollout
NUM_UPDATES = 500  # total PPO updates (increase for full training)
PPO_EPOCHS = 4  # epochs per update
MINIBATCH_SIZE = 512
LEARNING_RATE = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
ENT_COEF = 0.01
FALL_PENALTY = -10.0

# Load reward config from TOML (uses SPECIES and CURRENT_STAGE from config cell)
from environments.shared.config import load_stage_config

_stage_cfg = load_stage_config(SPECIES, CURRENT_STAGE)
_env_kw = _stage_cfg["env_kwargs"]
reward_cfg = {
    "forward_vel_weight": _env_kw.get("forward_vel_weight", 0.0),
    "alive_bonus": _env_kw.get("alive_bonus", 1.0),
    "energy_penalty_weight": _env_kw.get("energy_penalty_weight", 0.001),
    "posture_weight": _env_kw.get("posture_weight", 0.2),
}

print("Training config:")
print(f"  Species: {SPECIES}")
print(f"  Envs: {NUM_ENVS}")
print(f"  Rollout length: {ROLLOUT_LEN}")
print(f"  Updates: {NUM_UPDATES}")
print(f"  Total env steps: {NUM_ENVS * ROLLOUT_LEN * NUM_UPDATES:,}")
print(f"  Stage: {CURRENT_STAGE} ({_cfg['stage_names'].get(CURRENT_STAGE, '?')})")
print(f"  Reward config: {reward_cfg}")

In [ ]:
# Initialize batched environments
rng = jax.random.PRNGKey(42)

# Reset: create initial MJX data for all envs
mujoco.mj_resetData(mj_model, mj_data)
mujoco.mj_forward(mj_model, mj_data)
base_data = mjx.put_data(mj_model, mj_data)


# Replicate across batch with small perturbations
def init_env(rng):
    noise = jax.random.uniform(rng, (mj_model.nq - 7,), minval=-0.01, maxval=0.01)
    qpos = base_data.qpos.at[7:].add(noise)
    return base_data.replace(qpos=qpos)


rngs = jax.random.split(rng, NUM_ENVS)
env_batch = jax.vmap(init_env)(rngs)

# Forward pass to update derived quantities
env_batch = jax.jit(jax.vmap(mjx.forward, in_axes=(None, 0)))(mjx_model, env_batch)

print(f"Initialized {NUM_ENVS} parallel environments.")
print(f"qpos batch shape: {env_batch.qpos.shape}")

In [ ]:
# Optimizer
optimizer = optax.adam(LEARNING_RATE)
opt_state = optimizer.init(params)


# JIT-compiled PPO update step (with gradient norm and loss decomposition)
@jax.jit
def ppo_update(params, opt_state, obs, actions, log_probs, advantages, returns):
    (loss, aux), grads = jax.value_and_grad(nb_ppo_loss, has_aux=True)(
        params,
        obs,
        actions,
        log_probs,
        advantages,
        returns,
        clip_range=CLIP_RANGE,
        ent_coef=ENT_COEF,
    )
    grad_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree.leaves(grads)))
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss, aux, grad_norm


# JIT-compiled batched action sampling
@jax.jit
def batched_sample(params, obs_batch, rng):
    rngs = jax.random.split(rng, obs_batch.shape[0])
    return jax.vmap(nb_sample_action, in_axes=(None, 0, 0))(params, obs_batch, rngs)


# JIT-compiled batched observation + reward
@jax.jit
def batched_obs(data_batch):
    return jax.vmap(get_obs)(data_batch)


@jax.jit
def batched_reward(data_batch, action_batch):
    return jax.vmap(compute_reward, in_axes=(0, 0, None))(data_batch, action_batch, reward_cfg)


@jax.jit
def batched_terminated(data_batch):
    return jax.vmap(is_terminated)(data_batch)


# Reset helper: reset fallen envs to initial state
@jax.jit
def reset_fallen(env_batch, dones, rng):
    """Reset environments where done=True."""
    rngs = jax.random.split(rng, NUM_ENVS)
    fresh = jax.vmap(init_env)(rngs)
    fresh = jax.vmap(mjx.forward, in_axes=(None, 0))(mjx_model, fresh)

    # Select fresh state for done envs, keep existing for others
    def select(fresh_field, existing_field):
        expand = dones.reshape((-1,) + (1,) * (existing_field.ndim - 1))
        return jnp.where(expand, fresh_field, existing_field)

    return jax.tree.map(select, fresh, env_batch)


print("JIT functions ready (with gradient norm tracking).")

In [ ]:
# ---------- Main Training Loop ----------
# Enhanced with: gradient norm tracking, decomposed loss logging,
# periodic checkpointing, best-model tracking, and CSV log for post-hoc analysis.

import csv
import pickle

CHECKPOINT_FREQ = 50  # Save checkpoint every N updates

# Console log frequency based on VERBOSE:
# 0 = only final summary, 1 = every 20 updates (default), 2 = every update
_LOG_INTERVAL = {0: None, 1: 20, 2: 1}.get(VERBOSE, 20)

reward_history = []
loss_history = []
diagnostics_history = []  # Per-update detailed metrics

# Best model tracking
best_reward = -float("inf")
best_params = None
best_update = -1

# CSV log file for easy import into pandas/spreadsheets
csv_path = OUTPUT_DIR / f"{SPECIES}_jax_training_log.csv"
csv_fields = [
    "update",
    "reward",
    "total_loss",
    "policy_loss",
    "value_loss",
    "entropy",
    "approx_kl",
    "clip_fraction",
    "grad_norm",
    "mean_std",
    "steps",
    "sps",
    "fall_rate",
]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()

print(f"Starting training: {NUM_UPDATES} updates x {ROLLOUT_LEN} steps x {NUM_ENVS} envs")
print(f"Checkpoint frequency: every {CHECKPOINT_FREQ} updates")
print(f"CSV log: {csv_path}")
print("=" * 70)

t_start = time.time()

for update in range(NUM_UPDATES):
    # ---------- Collect rollout ----------
    all_obs, all_actions, all_log_probs, all_values = [], [], [], []
    all_rewards, all_dones = [], []

    for t in range(ROLLOUT_LEN):
        rng, rng_act, rng_reset = jax.random.split(rng, 3)

        obs = batched_obs(env_batch)
        actions, log_probs, values = batched_sample(params, obs, rng_act)

        # Step environments
        env_batch = batched_step(mjx_model, env_batch, actions)

        rewards = batched_reward(env_batch, actions)
        dones = batched_terminated(env_batch)

        # Add fall penalty
        rewards = rewards + dones.astype(jnp.float32) * FALL_PENALTY

        all_obs.append(obs)
        all_actions.append(actions)
        all_log_probs.append(log_probs)
        all_values.append(values)
        all_rewards.append(rewards)
        all_dones.append(dones.astype(jnp.float32))

        # Reset fallen envs
        env_batch = reset_fallen(env_batch, dones, rng_reset)

    # Stack rollout data: (T, NUM_ENVS, ...)
    obs_t = jnp.stack(all_obs)  # (T, N, obs_dim)
    act_t = jnp.stack(all_actions)  # (T, N, act_dim)
    lp_t = jnp.stack(all_log_probs)  # (T, N)
    val_t = jnp.stack(all_values)  # (T, N)
    rew_t = jnp.stack(all_rewards)  # (T, N)
    done_t = jnp.stack(all_dones)  # (T, N)

    # ---------- Compute advantages ----------
    advantages, returns = compute_gae(rew_t, val_t, done_t, GAMMA, GAE_LAMBDA)

    # Flatten: (T * N, ...)
    flat_obs = obs_t.reshape(-1, OBS_DIM)
    flat_act = act_t.reshape(-1, ACT_DIM)
    flat_lp = lp_t.reshape(-1)
    flat_adv = advantages.reshape(-1)
    flat_ret = returns.reshape(-1)

    # ---------- PPO update epochs ----------
    total_samples = flat_obs.shape[0]
    epoch_losses = []
    epoch_aux = []
    epoch_grad_norms = []

    for epoch in range(PPO_EPOCHS):
        rng, rng_perm = jax.random.split(rng)
        perm = jax.random.permutation(rng_perm, total_samples)

        for start in range(0, total_samples, MINIBATCH_SIZE):
            idx = perm[start : start + MINIBATCH_SIZE]
            mb_obs = flat_obs[idx]
            mb_act = flat_act[idx]
            mb_lp = flat_lp[idx]
            mb_adv = flat_adv[idx]
            mb_ret = flat_ret[idx]

            params, opt_state, loss, aux, grad_norm = ppo_update(
                params,
                opt_state,
                mb_obs,
                mb_act,
                mb_lp,
                mb_adv,
                mb_ret,
            )
            epoch_losses.append(float(loss))
            epoch_aux.append({k: float(v) for k, v in aux.items()})
            epoch_grad_norms.append(float(grad_norm))

    avg_reward = float(rew_t.mean())
    avg_loss = np.mean(epoch_losses)
    avg_grad_norm = np.mean(epoch_grad_norms)
    avg_aux = {k: np.mean([a[k] for a in epoch_aux]) for k in epoch_aux[0]}
    fall_rate = float(done_t.sum()) / (ROLLOUT_LEN * NUM_ENVS)

    reward_history.append(avg_reward)
    loss_history.append(avg_loss)
    diagnostics_history.append(
        {
            "reward": avg_reward,
            "loss": avg_loss,
            "grad_norm": avg_grad_norm,
            "fall_rate": fall_rate,
            **avg_aux,
        }
    )

    # Track best model
    if avg_reward > best_reward:
        best_reward = avg_reward
        best_params = jax.device_get(params)
        best_update = update

    # Write to CSV log
    elapsed = time.time() - t_start
    steps_done = (update + 1) * ROLLOUT_LEN * NUM_ENVS
    sps = steps_done / elapsed
    csv_writer.writerow(
        {
            "update": update,
            "reward": f"{avg_reward:.4f}",
            "total_loss": f"{avg_loss:.4f}",
            "policy_loss": f"{avg_aux['policy_loss']:.4f}",
            "value_loss": f"{avg_aux['value_loss']:.4f}",
            "entropy": f"{avg_aux['entropy']:.4f}",
            "approx_kl": f"{avg_aux['approx_kl']:.6f}",
            "clip_fraction": f"{avg_aux['clip_fraction']:.4f}",
            "grad_norm": f"{avg_grad_norm:.4f}",
            "mean_std": f"{avg_aux['mean_std']:.4f}",
            "steps": steps_done,
            "sps": f"{sps:.0f}",
            "fall_rate": f"{fall_rate:.4f}",
        }
    )
    csv_file.flush()

    # Console logging (respects VERBOSE setting)
    if _LOG_INTERVAL is not None and (update % _LOG_INTERVAL == 0 or update == NUM_UPDATES - 1):
        print(
            f"Update {update:4d}/{NUM_UPDATES}  "
            f"reward={avg_reward:+.3f}  loss={avg_loss:.4f}  "
            f"pi_loss={avg_aux['policy_loss']:.4f}  v_loss={avg_aux['value_loss']:.4f}  "
            f"entropy={avg_aux['entropy']:.3f}  kl={avg_aux['approx_kl']:.5f}  "
            f"grad={avg_grad_norm:.3f}  falls={fall_rate:.2%}  "
            f"SPS={sps:,.0f}"
        )

    # Periodic checkpointing
    if (update + 1) % CHECKPOINT_FREQ == 0:
        ckpt_path = OUTPUT_DIR / f"{SPECIES}_jax_checkpoint_{update + 1}.pkl"
        with open(ckpt_path, "wb") as f:
            pickle.dump(
                {
                    "params": jax.device_get(params),
                    "update": update + 1,
                    "reward_history": reward_history,
                    "loss_history": loss_history,
                },
                f,
            )
        if _LOG_INTERVAL is not None:
            print(f"  >>> Checkpoint saved: {ckpt_path}")

csv_file.close()

elapsed = time.time() - t_start
total_steps = NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS
print("=" * 70)
print(f"Done! {total_steps:,} steps in {elapsed:.1f}s ({total_steps / elapsed:,.0f} SPS)")
print(f"Best reward: {best_reward:+.4f} at update {best_update}")

# Save final trained parameters and full training history
params_path = OUTPUT_DIR / f"{SPECIES}_jax_params.pkl"
with open(params_path, "wb") as f:
    pickle.dump(
        {
            "params": jax.device_get(params),
            "best_params": best_params,
            "best_reward": best_reward,
            "best_update": best_update,
            "reward_history": reward_history,
            "loss_history": loss_history,
            "diagnostics_history": diagnostics_history,
        },
        f,
    )
print(f"Parameters and training history saved to: {params_path}")
print(f"Training log CSV saved to: {csv_path}")

## Stage Gate Evaluation

Run full evaluation episodes on CPU to check whether the curriculum gate
thresholds (min reward and min episode length) have been met. Both conditions
must pass before advancing to the next training stage.

In [ ]:
# ---------- Stage Gate Evaluation ----------
# Run full evaluation episodes on CPU MuJoCo to measure proper episode
# rewards and lengths, then check against the curriculum gate thresholds
# from the TOML config. Both min_avg_reward and min_avg_episode_length
# must be met before advancing to the next stage.

from environments.shared.config import load_stage_config

N_EVAL_EPISODES = 25

# Load gate thresholds from TOML config
stage_config = load_stage_config(SPECIES, CURRENT_STAGE)
curriculum = stage_config.get("curriculum_kwargs", {})
gate_min_reward = curriculum.get("min_avg_reward", -float("inf"))
gate_min_length = curriculum.get("min_avg_episode_length", 0)

print(f"Stage {CURRENT_STAGE} curriculum gate thresholds:")
print(f"  min_avg_reward:         {gate_min_reward}")
print(f"  min_avg_episode_length: {gate_min_length}")
print(f"\nRunning {N_EVAL_EPISODES} evaluation episodes on CPU...")

# Use best params from training
eval_params = best_params if best_params is not None else jax.device_get(params)

eval_rewards = []
eval_lengths = []

for ep in range(N_EVAL_EPISODES):
    mujoco.mj_resetData(mj_model, mj_data)
    # Small perturbation for variety
    mj_data.qpos[7:] += np.random.uniform(-0.01, 0.01, size=mj_data.qpos[7:].shape)
    mujoco.mj_forward(mj_model, mj_data)

    ep_reward = 0.0
    for step in range(MAX_EPISODE_STEPS):
        cpu_data = mjx.put_data(mj_model, mj_data)
        obs = get_obs(cpu_data)

        mean, log_std, value = network.apply(eval_params, obs)
        action = jnp.clip(mean, -1.0, 1.0)

        ctrl = np.array(scale_action(action))
        mj_data.ctrl[:] = ctrl
        for _ in range(FRAME_SKIP):
            mujoco.mj_step(mj_model, mj_data)

        cpu_data = mjx.put_data(mj_model, mj_data)
        r = float(compute_reward(cpu_data, action, reward_cfg))
        ep_reward += r

        body_z = mj_data.xpos[ROOT_BODY_ID, 2]
        if body_z < HEALTHY_Z_MIN or body_z > HEALTHY_Z_MAX:
            break

    ep_length = step + 1
    eval_rewards.append(ep_reward)
    eval_lengths.append(ep_length)

mean_reward = np.mean(eval_rewards)
mean_length = np.mean(eval_lengths)
std_reward = np.std(eval_rewards)
std_length = np.std(eval_lengths)

print(f"\nEvaluation results ({N_EVAL_EPISODES} episodes):")
print(f"  Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"  Mean length: {mean_length:.1f} +/- {std_length:.1f}")

# Check gate conditions
gate_failures = []
if mean_reward < gate_min_reward:
    gate_failures.append(f"mean reward {mean_reward:.2f} < {gate_min_reward}")
if mean_length < gate_min_length:
    gate_failures.append(f"mean episode length {mean_length:.1f} < {gate_min_length}")

if gate_failures:
    print(f"\n*** STAGE {CURRENT_STAGE} GATE NOT PASSED ***")
    for f in gate_failures:
        print(f"  - {f}")
    print("Do NOT proceed to the next stage. Re-train with more updates or adjusted hyperparameters.")
    raise RuntimeError(
        f"Stage {CURRENT_STAGE} curriculum gate not passed: {'; '.join(gate_failures)}. "
        "Both min_avg_reward and min_avg_episode_length must be met before advancing."
    )
else:
    print(f"\n*** STAGE {CURRENT_STAGE} GATE PASSED ***")
    print("You may proceed to the next training stage.")

## Save Results

Save a `summary.json` for this JAX/MJX training run.

In [ ]:
import json

repo_root = Path("/content/mesozoic-labs")
results_dir = repo_root / "results" / SPECIES / "ppo"
results_dir.mkdir(parents=True, exist_ok=True)

def _format_duration_hms(seconds):
    """Format seconds as H:MM:SS."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h}:{m:02d}:{s:02d}"

summary = {
    "species": SPECIES,
    "algorithm": "PPO (JAX/MJX)",
    "hardware": "Google Colab GPU",
    "seed": 42,
    "date": time.strftime("%Y-%m-%d"),
    "stages": {
        "1": {
            "name": "balance",
            "timesteps": total_steps,
            "avg_reward": round(best_reward, 2),
            "training_time_seconds": round(elapsed, 1),
            "training_time": _format_duration_hms(elapsed),
        }
    },
    "total_timesteps": total_steps,
    "total_training_time_seconds": round(elapsed, 1),
    "total_training_time": _format_duration_hms(elapsed),
    "final_avg_reward": round(best_reward, 2),
    "steps_per_second": round(total_steps / elapsed),
    "num_envs": NUM_ENVS,
    "num_updates": NUM_UPDATES,
}

summary_path = results_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print(f"Results summary saved to: {summary_path}")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Reward
ax = axes[0, 0]
ax.plot(reward_history, "b-", alpha=0.3, label="per-update")
window = min(20, len(reward_history) // 4 + 1)
if window > 1:
    smoothed = np.convolve(reward_history, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(reward_history)), smoothed, "b-", linewidth=2, label=f"{window}-update avg")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Mean Reward")
ax.set_title("Reward")
ax.legend()
ax.grid(True, alpha=0.3)

# Total loss
ax = axes[0, 1]
ax.plot(loss_history, "r-", alpha=0.5)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Loss")
ax.set_title("Total Loss")
ax.grid(True, alpha=0.3)

# Gradient norm
ax = axes[0, 2]
grad_norms = [d["grad_norm"] for d in diagnostics_history]
ax.plot(grad_norms, "g-", alpha=0.5)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Gradient Norm")
ax.set_title("Gradient Norm")
ax.grid(True, alpha=0.3)

# Policy loss vs Value loss
ax = axes[1, 0]
pi_losses = [d["policy_loss"] for d in diagnostics_history]
v_losses = [d["value_loss"] for d in diagnostics_history]
ax.plot(pi_losses, "b-", alpha=0.5, label="Policy loss")
ax.plot(v_losses, "r-", alpha=0.5, label="Value loss")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Loss")
ax.set_title("Policy vs Value Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Entropy and KL divergence
ax = axes[1, 1]
entropies = [d["entropy"] for d in diagnostics_history]
ax.plot(entropies, "purple", alpha=0.7, label="Entropy")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Entropy")
ax.set_title("Policy Entropy")
ax2 = ax.twinx()
kls = [d["approx_kl"] for d in diagnostics_history]
ax2.plot(kls, "orange", alpha=0.7, label="Approx KL")
ax2.set_ylabel("Approx KL")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")
ax.grid(True, alpha=0.3)

# Fall rate
ax = axes[1, 2]
fall_rates = [d["fall_rate"] for d in diagnostics_history]
ax.plot(fall_rates, "brown", alpha=0.5)
if window > 1:
    smoothed_falls = np.convolve(fall_rates, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(fall_rates)), smoothed_falls, "brown", linewidth=2)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Fall Rate")
ax.set_title("Episode Termination Rate")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.suptitle(f"{SPECIES.title()} JAX/MJX Training Diagnostics", fontsize=14, fontweight="bold")
plt.tight_layout()
curve_path = OUTPUT_DIR / f"{SPECIES}_jax_training_curves.png"
plt.savefig(curve_path, dpi=150)
print(f"Training curves saved to: {curve_path}")
plt.show()

## 8. Record Training Video

Record a video of the best trained policy (highest reward during training) using
the CPU MuJoCo renderer. The JAX policy is evaluated deterministically (using the
action mean).

In [ ]:
try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Install with: pip install mediapy")

if _HAS_MEDIAPY:
    # Use the best model (highest reward during training) for video recording
    video_params = best_params if best_params is not None else jax.device_get(params)
    print(f"Recording video with best model (update {best_update}, reward {best_reward:+.4f})")

    # Set up CPU-based MuJoCo renderer
    renderer = mujoco.Renderer(mj_model, height=480, width=640)

    # Reset environment
    mujoco.mj_resetData(mj_model, mj_data)
    mujoco.mj_forward(mj_model, mj_data)

    frames = []
    episode_reward = 0.0

    for step in range(MAX_EPISODE_STEPS):
        # Get observation from CPU data
        cpu_data = mjx.put_data(mj_model, mj_data)
        obs = get_obs(cpu_data)

        # Get deterministic action (use mean, no noise)
        mean, log_std, value = network.apply(video_params, obs)
        action = jnp.clip(mean, -1.0, 1.0)

        # Apply action to CPU sim
        ctrl = np.array(scale_action(action))
        mj_data.ctrl[:] = ctrl
        for _ in range(FRAME_SKIP):
            mujoco.mj_step(mj_model, mj_data)

        # Render frame
        renderer.update_scene(mj_data)
        frames.append(renderer.render())

        # Compute reward
        cpu_data = mjx.put_data(mj_model, mj_data)
        r = float(compute_reward(cpu_data, action, reward_cfg))
        episode_reward += r

        # Check termination
        body_z = mj_data.xpos[ROOT_BODY_ID, 2]
        if body_z < HEALTHY_Z_MIN or body_z > HEALTHY_Z_MAX:
            break

    renderer.close()

    video_path = str(OUTPUT_DIR / f"{SPECIES}_jax_mjx_training.mp4")
    mediapy.write_video(video_path, frames, fps=50)
    print(f"Episode reward: {episode_reward:.2f} | {len(frames)} frames")
    print(f"Saved to: {video_path}")
    mediapy.show_video(frames, fps=50)

## 9. Next Steps

To continue with curriculum training, change `CURRENT_STAGE` in the configuration
cell above and re-run from there. The reward config is loaded automatically from the
TOML files in `configs/<species>/`:

```python
CURRENT_STAGE = 2  # or 3
```

The policy parameters carry over automatically between stages.
After each stage, re-run the Stage Gate Evaluation cell to verify the curriculum
thresholds are met, then re-run the video recording cell to capture the new behavior.

To train a different species, change `SPECIES` in the configuration cell and
restart from the beginning.